In [0]:
# bronze schema name
bronze_schema_name = 'ws_azuredatabricks.bronze'

Transform diagnosis table

In [0]:
# read bronze diagnosis table 
bronze_table_name = 'diagnosis'
diagnosis_df = spark.table(f'{bronze_schema_name}.{bronze_table_name}')

display(diagnosis_df)

diagnosis_code,diagnosis_desc
D001,Hypertension
D002,Diabetes Type 2
D003,Chest Pain
D004,Asthma
D005,Kidney Infection


In [0]:
# handling records with null values

diagnosis_df_row_count = diagnosis_df.count()
missing_code_row_count = diagnosis_df.dropna().count()

print(f'Rows with null values dropped : {diagnosis_df_row_count - missing_code_row_count}')

diagnosis_df = diagnosis_df.dropna()
display(diagnosis_df)

Rows with null values dropped : 0


diagnosis_code,diagnosis_desc
D001,Hypertension
D002,Diabetes Type 2
D003,Chest Pain
D004,Asthma
D005,Kidney Infection


In [0]:
# drop exact duplicate record, if any
duplicate_rows_count = diagnosis_df.dropDuplicates().count()

print(f'Duplicate rows dropped : {diagnosis_df_row_count - duplicate_rows_count}')

diagnosis_df = diagnosis_df.dropDuplicates()
display(diagnosis_df)

Duplicate rows dropped : 0


diagnosis_code,diagnosis_desc
D001,Hypertension
D002,Diabetes Type 2
D003,Chest Pain
D004,Asthma
D005,Kidney Infection


In [0]:
# similar code but different description, if any 
from pyspark.sql.functions import *

conflicting_codes_df = (
    diagnosis_df
    .groupBy("diagnosis_code")
    .agg(countDistinct(lower(trim(col("diagnosis_desc")))).alias("distinct_description_count"))
    .filter(col("distinct_description_count") > 1)
)

# if count > 0 then reach out to the client to resolve it
print(f'Conflicting diagnosis code count: {conflicting_codes_df.count()}')

if conflicting_codes_df.count() > 0:
    display(conflicting_codes_df)


Conflicting diagnosis code count: 0


In [0]:
# similar description but different code, if any 
conflicting_desc_df = (
    diagnosis_df
    .groupBy(lower(trim(col("diagnosis_desc"))))
    .agg(countDistinct(col('diagnosis_code')).alias("distinct_code_count"))
    .filter(col("distinct_code_count") > 1)
)

# if count > 0 then reach out to the client to resolve it
print(f'Conflicting diagnosis description count: {conflicting_desc_df.count()}')

if conflicting_desc_df.count() > 0:
    display(conflicting_desc_df)

Conflicting diagnosis description count: 0


In [0]:
# write the cleaned diagnosis table
silver_schema_name = 'ws_azuredatabricks.silver'

diagnosis_df.write.mode('overwrite').saveAsTable(f'{silver_schema_name}.{bronze_table_name}')

Transform hospital table

In [0]:
# read bronze hospital table
bronze_table_name = 'hospital'

hospital_df = spark.table(f'{bronze_schema_name}.{bronze_table_name}')
display(hospital_df)

hospital_id,hospital_name,city,bed_count
H001,Apollo Main Hospital,Chennai,850
H002,Fortis Healthcare Delhi,Delhi,600
H003,Kokilaben Dhirubhai Hospital,Mumbai,700
H004,Manipal Hospital,Bangalore,400
H005,Yashoda Hospital,Hyderabad,450


In [0]:
# drop rows with any NULL value
hospital_df_row_count = hospital_df.count()
row_count_after_null_drop = hospital_df.dropna().count()

print(f'Rows count afte null drop : {hospital_df_row_count - row_count_after_null_drop}')
hospital_df = hospital_df.dropna()

display(hospital_df)

Rows count afte null drop : 0


hospital_id,hospital_name,city,bed_count
H001,Apollo Main Hospital,Chennai,850
H002,Fortis Healthcare Delhi,Delhi,600
H003,Kokilaben Dhirubhai Hospital,Mumbai,700
H004,Manipal Hospital,Bangalore,400
H005,Yashoda Hospital,Hyderabad,450


In [0]:
# drop exacr duplicate records
duplicate_row_count = hospital_df.dropDuplicates().count()
print(f'Duplicate rows dropped : {hospital_df_row_count - duplicate_row_count}')

hospital_df = hospital_df.dropDuplicates()
display(hospital_df)

Duplicate rows dropped : 0


hospital_id,hospital_name,city,bed_count
H001,Apollo Main Hospital,Chennai,850
H002,Fortis Healthcare Delhi,Delhi,600
H003,Kokilaben Dhirubhai Hospital,Mumbai,700
H004,Manipal Hospital,Bangalore,400
H005,Yashoda Hospital,Hyderabad,450


In [0]:
# similar hospital_id
conflicting_hospital_id_df =(
    hospital_df
    .groupBy(col('hospital_id'))
    .agg(
        count('*').alias('Record Count')
    )
    .filter(col('Record Count') > 1)
)

print(f'Hospital with similar id : {conflicting_hospital_id_df.count()}')

if conflicting_hospital_id_df.count() > 0:
    display(conflicting_hospital_id_df)

Hospital with similar id : 0


In [0]:
# write cleaned hospital table
silver_schema_name = 'ws_azuredatabricks.silver'

hospital_df.write.mode('overwrite').saveAsTable(f'{silver_schema_name}.{bronze_table_name}')

Transform patient table

In [0]:
# read bronze patient table
bronze_table_name = 'patient'

patient_df = spark.table(f'{bronze_schema_name}.{bronze_table_name}')
display(patient_df)

patient_id,first_name,last_name,gender,dob,city
P001,Aarav,Sharma,M,1980-02-14,Mumbai
P002,Riya,Verma,F,1975-07-22,Delhi
P003,Kabir,Menon,M,1990-11-02,Bangalore
P004,Sneha,Rao,F,1988-09-19,Hyderabad
P005,Vikram,Singh,M,1965-05-05,Chennai


In [0]:
# drop row with any NULL values

patient_df_row_count = patient_df.count()
row_count_after_null_drop = patient_df.dropna().count()

print(f'Rows with NULL values dropped : {patient_df_row_count - row_count_after_null_drop}')

patient_df = patient_df.dropna()
display(patient_df)

Rows with NULL values dropped : 0


patient_id,first_name,last_name,gender,dob,city
P001,Aarav,Sharma,M,1980-02-14,Mumbai
P002,Riya,Verma,F,1975-07-22,Delhi
P003,Kabir,Menon,M,1990-11-02,Bangalore
P004,Sneha,Rao,F,1988-09-19,Hyderabad
P005,Vikram,Singh,M,1965-05-05,Chennai


In [0]:
# drop exact duplicate records
duplicate_row_count = patient_df.dropDuplicates().count()
print(f'Duplicate rows dropped : {patient_df_row_count - duplicate_row_count}')

patient_df = patient_df.dropDuplicates()
display(patient_df)

Duplicate rows dropped : 0


patient_id,first_name,last_name,gender,dob,city
P001,Aarav,Sharma,M,1980-02-14,Mumbai
P002,Riya,Verma,F,1975-07-22,Delhi
P003,Kabir,Menon,M,1990-11-02,Bangalore
P004,Sneha,Rao,F,1988-09-19,Hyderabad
P005,Vikram,Singh,M,1965-05-05,Chennai


In [0]:
# DOB format
patient_df = patient_df.withColumn('dob', date_format(col('dob'), 'yyyy-MM-dd'))
display(patient_df)

patient_id,first_name,last_name,gender,dob,city
P001,Aarav,Sharma,M,1980-02-14,Mumbai
P002,Riya,Verma,F,1975-07-22,Delhi
P003,Kabir,Menon,M,1990-11-02,Bangalore
P004,Sneha,Rao,F,1988-09-19,Hyderabad
P005,Vikram,Singh,M,1965-05-05,Chennai


In [0]:
# conflicting patient id. Runs after dropDuplicates.

conflicting_patient_id_df =(
    patient_df
    .groupBy(col('patient_id'))
    .agg(
        count('*').alias('Patient Count')
    )
    .filter(col('Patient Count') > 1)
)

print(f'Patient with similar id : {conflicting_patient_id_df.count()}')

if conflicting_patient_id_df.count() > 0:
    display(conflicting_patient_id_df)

Patient with similar id : 0


In [0]:
# write cleaned patient table
silver_schema_name = 'ws_azuredatabricks.silver'

patient_df.write.mode('overwrite').saveAsTable(f'{silver_schema_name}.{bronze_table_name}')

Transform visit table

In [0]:
# read bronze visit table 
bronze_table_name = 'visit'

visit_df = spark.table(f'{bronze_schema_name}.{bronze_table_name}')
display(visit_df)

visit_id,patient_id,hospital_id,admission_date,discharge_date,diagnosis_code,cost
V1001,P001,H003,2025-01-01,2025-01-05,D003,12000
V1002,P001,H003,2025-01-20,2025-01-22,D003,9000
V1003,P002,H002,2025-02-10,2025-02-15,D001,18000
V1004,P002,H002,2025-04-01,2025-04-05,D001,17000
V1005,P003,H004,2025-03-12,2025-03-18,D005,22000
V1006,P003,H004,2025-03-28,2025-03-31,D005,16000
V1007,P004,H005,2025-01-15,2025-01-20,D004,14000
V1008,P005,H001,2025-02-25,2025-03-02,D002,20000


In [0]:
# drop null records
visit_df_row_count = visit_df.count()
row_count_after_null_drop = visit_df.dropna().count()

print(f'Rows with NULL values dropped : {visit_df_row_count - row_count_after_null_drop}')

visit_df = visit_df.dropna()
display(visit_df)

Rows with NULL values dropped : 0


visit_id,patient_id,hospital_id,admission_date,discharge_date,diagnosis_code,cost
V1001,P001,H003,2025-01-01,2025-01-05,D003,12000
V1002,P001,H003,2025-01-20,2025-01-22,D003,9000
V1003,P002,H002,2025-02-10,2025-02-15,D001,18000
V1004,P002,H002,2025-04-01,2025-04-05,D001,17000
V1005,P003,H004,2025-03-12,2025-03-18,D005,22000
V1006,P003,H004,2025-03-28,2025-03-31,D005,16000
V1007,P004,H005,2025-01-15,2025-01-20,D004,14000
V1008,P005,H001,2025-02-25,2025-03-02,D002,20000


In [0]:
# admission and discharge date format
visit_df = visit_df.withColumn('admission_date', date_format(col('admission_date'), 'yyyy-MM-dd'))
visit_df = visit_df.withColumn('discharge_date', date_format(col('discharge_date'), 'yyyy-MM-dd'))
display(visit_df)

visit_id,patient_id,hospital_id,admission_date,discharge_date,diagnosis_code,cost
V1001,P001,H003,2025-01-01,2025-01-05,D003,12000
V1002,P001,H003,2025-01-20,2025-01-22,D003,9000
V1003,P002,H002,2025-02-10,2025-02-15,D001,18000
V1004,P002,H002,2025-04-01,2025-04-05,D001,17000
V1005,P003,H004,2025-03-12,2025-03-18,D005,22000
V1006,P003,H004,2025-03-28,2025-03-31,D005,16000
V1007,P004,H005,2025-01-15,2025-01-20,D004,14000
V1008,P005,H001,2025-02-25,2025-03-02,D002,20000


In [0]:
# Valid IDs from the cleaned (silver) reference tables
valid_patient_ids_df   = spark.table(f'{silver_schema_name}.patient').select("patient_id").distinct().withColumn("patient_exists", lit(True))
valid_hospital_ids_df  = spark.table(f'{silver_schema_name}.hospital').select("hospital_id").distinct().withColumn("hospital_exists", lit(True))
valid_diagnosis_ids_df = spark.table(f'{silver_schema_name}.diagnosis').select("diagnosis_code").distinct().withColumn("diagnosis_exists", lit(True))

# Attach a "found / not found" flag for each reference
visit_checked_df = (
    visit_df
    .join(valid_patient_ids_df,   on="patient_id",     how="left")
    .join(valid_hospital_ids_df,  on="hospital_id",    how="left")
    .join(valid_diagnosis_ids_df, on="diagnosis_code", how="left")
)


all_references_found = (
    col("patient_exists").isNotNull()
    & col("hospital_exists").isNotNull()
    & col("diagnosis_exists").isNotNull()
)

# Valid visits → silver
visit_valid_df = (
    visit_checked_df
    .filter(all_references_found)
    .drop("patient_exists", "hospital_exists", "diagnosis_exists")
)


# Orphan visits → quarantine, with the exact reason
visit_quarantine_df = (
    visit_checked_df
    .filter(~all_references_found)
    .withColumn("quarantine_reason", concat_ws(", ",
        when(col("patient_exists").isNull(),   lit("unknown patient_id")),
        when(col("hospital_exists").isNull(),  lit("unknown hospital_id")),
        when(col("diagnosis_exists").isNull(), lit("unknown diagnosis_code")),
    ))
    .withColumn("quarantined_at", current_timestamp())
    .drop("patient_exists", "hospital_exists", "diagnosis_exists")
)

print(f"Valid visits: {visit_valid_df.count()} | Orphan visits: {visit_quarantine_df.count()}")

visit_df = visit_valid_df
visit_df.write.mode("overwrite").saveAsTable(f"{silver_schema_name}.visit")
visit_quarantine_df.write.mode("append").saveAsTable(f"{silver_schema_name}.visit_quarantine")

Valid visits: 8 | Orphan visits: 0
